This notebook largely follows the approach of ["An NLP-based approach to assessing a company's maturity level in the digital era" (2024) by Romano, Sperlì, and Vignali](https://www.sciencedirect.com/science/article/pii/S0957417424011588)

Extra pre-requisites:
* Create a Hugging Face (HF) account
* Create a HF access token with at least read access
* Save the token value within Google Colab secrets (secret name: "HF_TOKEN") or within ??? if running this notebook locally.

In [4]:
import math
import os
import pandas as pd
import re
from sklearn.feature_extraction.text import CountVectorizer
import string
import sys
import transformers

NLTK Downloader
---------------------------------------------------------------------------
    d) Download   l) List    u) Update   c) Config   h) Help   q) Quit
---------------------------------------------------------------------------
Downloader> d

Download which package (l=list; x=cancel)?
  Identifier> all


       | 
       | Downloading package abc to /root/nltk_data...
       |   Unzipping corpora/abc.zip.
       | Downloading package alpino to /root/nltk_data...
       |   Unzipping corpora/alpino.zip.
       | Downloading package averaged_perceptron_tagger to
       |     /root/nltk_data...
       |   Unzipping taggers/averaged_perceptron_tagger.zip.
       | Downloading package averaged_perceptron_tagger_eng to
       |     /root/nltk_data...
       |   Unzipping taggers/averaged_perceptron_tagger_eng.zip.
       | Downloading package averaged_perceptron_tagger_ru to
       |     /root/nltk_data...
       |   Unzipping taggers/averaged_perceptron_tagger_ru.zip.
       | Downloading package averaged_perceptron_tagger_rus to
       |     /root/nltk_data...
       |   Unzipping taggers/averaged_perceptron_tagger_rus.zip.
       | Downloading package basque_grammars to /root/nltk_data...
       |   Unzipping grammars/basque_grammars.zip.
       | Downloading package bcp47 to /root/nltk_d


---------------------------------------------------------------------------
    d) Download   l) List    u) Update   c) Config   h) Help   q) Quit
---------------------------------------------------------------------------
Downloader> d

Download which package (l=list; x=cancel)?
  Identifier> all


       | 
       | Downloading package abc to /root/nltk_data...
       |   Package abc is already up-to-date!
       | Downloading package alpino to /root/nltk_data...
       |   Package alpino is already up-to-date!
       | Downloading package averaged_perceptron_tagger to
       |     /root/nltk_data...
       |   Package averaged_perceptron_tagger is already up-to-date!
       | Downloading package averaged_perceptron_tagger_eng to
       |     /root/nltk_data...
       |   Package averaged_perceptron_tagger_eng is already up-to-
       |       date!
       | Downloading package averaged_perceptron_tagger_ru to
       |     /root/nltk_data...
       |   Package averaged_perceptron_tagger_ru is already up-to-
       |       date!
       | Downloading package averaged_perceptron_tagger_rus to
       |     /root/nltk_data...
       |   Package averaged_perceptron_tagger_rus is already up-to-
       |       date!
       | Downloading package basque_grammars to /root/nltk_data...
     


---------------------------------------------------------------------------
    d) Download   l) List    u) Update   c) Config   h) Help   q) Quit
---------------------------------------------------------------------------
Downloader> q


In [ ]:
import nltk
nltk.download()
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import PunktSentenceTokenizer, RegexpTokenizer

In [62]:
# Comment out all but the appropriate option
where_running = "Google Colab"
# where_running = "Local Machine"

if where_running == "Local Machine":
  module_path = os.path.abspath(os.path.join('..'))
  if not module_path in sys.path:
      sys.path.insert(0, module_path)
elif where_running == "Google Colab":
  dirpath = '/content/digi-inno-road-prod'
  if not os.path.isdir(dirpath):
    # TODO git pull
    !git clone https://github.com/roughhawkbit/digi-inno-road-prod.git
  sys.path.insert(0,dirpath)

from innoprod.sheet_tools import get_sheet_dfs
from innoprod.text_analysis.chunking_tools import chunk_text_sentencewise
from innoprod.wrangling.msyh_data_sharing import wrangle_roadmaps
from innoprod.wrangling.wrangling_tools import is_non_empty

In [5]:
data = get_sheet_dfs()
roadmaps_df = wrangle_roadmaps(data['Roadmaps'])

In [6]:
cols = [
    'Summary review of Edge Digital diagnostic report & current state and key improvement areas',
    'What are the internal barriers to growth? How do you intend to finance future growth? Are there sufficient leadership and management skills in the business to achieve your growth? What opportunities do you have to expand into new markets?',
    'Level of current Strategic Digital Skills/knowledge in the business',
    'Level of current Technical Digital Skills/knowledge in the business',
    'Whether the business is already investing/adopting/utilising Industry 4.0 Technologies, with examples',
    'Summary of the identified problems, including Gap Analysis',
    'Key potential industry 4.0 solutions to address the identified problems/gaps',
    'Recommended Action Plan to utilise Industry 4.0 Technology',
    'Overview of qualitative benefits of recommended Action Plan (positive/negative)',
    'Skills and other requirements that will be needed to successfully implement the recommended Action Plan',
    'What has been your overall opinion of the support you have received in this programme? (Add comments)'
]

In [ ]:
responses_df = roadmaps_df[['Client ID'] + cols].melt(id_vars=['Client ID'], value_vars=cols, var_name='Question', value_name='Response')
responses_df = responses_df[is_non_empty(responses_df['Response'])]

responses_df

In [ ]:
sent_tokenizer = PunktSentenceTokenizer()

sentences_df = responses_df.copy()
sentences_df['Sentences'] = sentences_df.apply(lambda row: sent_tokenizer.tokenize(row['Response']), axis=1)
sentences_df = sentences_df[['Client ID', 'Question', 'Sentences']].explode('Sentences').reset_index().rename(columns={'index' : 'Sentence Number', 'Sentences' : 'Sentence'})
sentences_df['Sentence Number'] = sentences_df.groupby('Sentence Number').cumcount() + 1
sentences_df

# Preprocessing

In [9]:
lemmatizer = WordNetLemmatizer()

# Found initially that some lemmatized stop words slipped through the elimination step, so including these in order to catch both the original and the lemmatized forms.
# For example, "has" and "have" are in the stop words list, but these are lemmatized to "ha" which is not.
lemmatized_stops = set([lemmatizer.lemmatize(word) for word in stopwords.words('english')] + stopwords.words('english'))

In [10]:
# These acronyms and abbreviations will be expanded into their full forms.
acronyms_abbreviations = {
    'I4.0': 'industry 4.0',
    'tech': 'technology',
}

# Here we ensure that "4.0" is kept together as a word token: this should occur often together with "Industry".
# TODO confirm that this is working as expected (or is the CountVectorizer undoing the preprocessing?)
regexpr = "4\\.0|[\\w']+"
re_tokenizer = RegexpTokenizer(regexpr)

In [11]:
def preprocess_text(text):
    cleaned_text = text.replace("[REDACTED]", "")
    for short_form, full_form in acronyms_abbreviations.items():
        matches = re.findall(f'({short_form})[\\s|{re.escape(string.punctuation)}]', cleaned_text)
        for match in matches:
            cleaned_text = re.sub(re.escape(match), full_form, cleaned_text, flags=re.IGNORECASE)
    # Skipped removing punctuation as tokenization already does this.
    cleaned_text = cleaned_text.lower()
    tokenized_answer = re_tokenizer.tokenize(cleaned_text)
    lemmatized_answer = [lemmatizer.lemmatize(word) for word in tokenized_answer]
    final_answer = [lemma for lemma in lemmatized_answer if lemma not in lemmatized_stops]
    return " ".join(final_answer)

In [12]:
# responses_df['Cleaned Response'] = responses_df.apply(lambda row: preprocess_text(row['Response']), axis=1)
sentences_df['Cleaned Sentence'] = sentences_df.apply(lambda row: preprocess_text(row['Sentence']), axis=1)

In [13]:
vectorizer = CountVectorizer(analyzer='word', ngram_range=(1,1))
transform_array = vectorizer.fit_transform(sentences_df['Cleaned Sentence'])
feature_names = vectorizer.get_feature_names_out()

In [14]:
features_df = pd.DataFrame({
    "Feature": feature_names,
    "Count": transform_array.sum(axis=0).getA1()
})

In [ ]:
features_df.sort_values('Count', ascending=False)

Inspection of the 30 most frequent words[1] suggests that this approach will not yield many useful concepts. Instead, we look at bigrams (word pairs) and trigrams (triples) as manual inspection of a sample of responses suggested this may be more fruitful (e.g., "digital strategy" rather than "digital" or "strategy" occuring without the other).

[1] Excluding stop words

In [16]:
vectorizer = CountVectorizer(analyzer='word', ngram_range=(1,3))
transform_array = vectorizer.fit_transform(sentences_df['Cleaned Sentence'])
feature_names = vectorizer.get_feature_names_out()

In [17]:
features_df = pd.DataFrame({
    "Feature": feature_names,
    "Count": transform_array.sum(axis=0).getA1()
})
features_df['N-Gram'] = features_df['Feature'].apply(lambda feature: len(re_tokenizer.tokenize(feature)))

In [ ]:
features_df.sort_values('Count', ascending=False)

# Expert supervision
By manually inspecting the N-grams above, starting with the most frequent and working down the list, the following N-grams were identified as significant and grouped into concepts.

(So far assessed the first 6 pages - 25 results per page)

## TODO
* Visualise how these concepts map onto the different questions... would expect to see "recent innovation" to be within "Whetehr the business is already investing/adopting/utilising Industry 4.0 Tech..." and/or "Summary of the identified problems..."

In [19]:
concepts = {
    'Access to finance': ['investment', 'cost', 'invested'],
	  'Access to skills': ['skill', 'knowledge', 'training'],
    'Strategic alignment': ['strategy', 'productivity', 'efficiency', 'digital strategy', 'digital transformation', 'data collection', 'business growth', 'strategic'],
    'Recent innovation': [],
    'Attitudes to innovation': ['understanding', 'understand'],
    'External support/advice': []
}

# Information extraction

In [ ]:
model_name = "pborchert/BusinessBERT"
pipe = transformers.pipeline("fill-mask", model=model_name)
tokenizer = transformers.AutoTokenizer.from_pretrained(model_name)

In [21]:
recent_innovation_questions = ['Summary of the identified problems, including Gap Analysis', 'Whether the business is already investing/adopting/utilising Industry 4.0 Technologies, with examples']
prompt = f"This company has a {tokenizer.special_tokens_map['mask_token']} record of recent innovation and technology adoption."

Test case: try approach with just one firm and one concept

In [ ]:
client_1_df = responses_df[responses_df['Client ID'] == "f3fff05d-1a28-e8f3-c5f6-670d1d0797e3"]
client_1_recent_innovation = " ".join(client_1_df[client_1_df['Question'].isin(recent_innovation_questions)]['Response'].tolist())
client_1_recent_innovation

In [23]:
mask_prompt = f"{client_1_recent_innovation}. {prompt}"

In [ ]:
mask_output = pipe(mask_prompt)
output_df = pd.DataFrame(mask_output)
output_df

Now generalise across all firms for the same concept. Note that we need to be very careful about response lengths, as the maximum number of tokens that a BERT analysis can handle in one query is 512. Some individual responses breach this limit.

In [ ]:
recent_innovation_df = responses_df[responses_df['Question'].isin(recent_innovation_questions)].groupby(['Client ID'])['Response'].apply('. '.join).to_frame().reset_index()
recent_innovation_df = recent_innovation_df.rename(columns={'Response': 'Combined Responses'})
#  recent_innovation_df['Prompt'] = recent_innovation_df['Combined Responses'].apply(lambda responses: f"{responses}. {prompt}")
# recent_innovation_df['Predictions'] = recent_innovation_df['Prompt'].apply(lambda mask_prompt: {output['token_str']: output['score'] for output in pipe(mask_prompt)})
recent_innovation_df['Combined Responses Len'] = recent_innovation_df['Combined Responses'].apply(lambda response: len(tokenizer.tokenize(response)))
recent_innovation_df

In [48]:
n_prompt_tokens = len(tokenizer.tokenize(prompt))
n_prompt_tokens

13

In [66]:
for response in recent_innovation_df['Combined Responses']:
  try:
    chunk_text_sentencewise(response, 512-n_prompt_tokens)
  except ValueError as e:
    problem_response = response

In [ ]:
problem_response

In [93]:
from innoprod.text_analysis.chunking_tools import split_token_dict # Explicitly import split_token_dict to use in modified function
from collections import OrderedDict
import math
sentence_tokens = sent_tokenizer.tokenize(problem_response)
token_dict = OrderedDict()
for sentence in sentence_tokens:
  n_words = len(tokenizer.tokenize(sentence))
  if n_words > 0:
    token_dict[sentence] = n_words

max_len = 512-n_prompt_tokens

total_len = sum(token_dict.values())
min_n_chunks = math.ceil(total_len / max_len)
target_chunk_len = int(round(total_len / min_n_chunks, 0))
new_dict = OrderedDict()
counter = 0
while counter < target_chunk_len:
  if next(iter(token_dict.values())) + counter > max_len:
    break
  sentence, token_len = token_dict.popitem(last=False)
  counter += token_len
  new_dict[sentence] = token_len


In [ ]:
new_dict

In [ ]:
max_len

In [ ]:
token_dict

In [103]:
text = problem_response
max_words = 512-n_prompt_tokens

sent_tokenizer = PunktSentenceTokenizer()
word_tokenizer = RegexpTokenizer(r'\w+')
sentence_tokens = sent_tokenizer.tokenize(text)
od = OrderedDict()
for sentence in sentence_tokens:
  n_words = len(word_tokenizer.tokenize(sentence))
  if n_words > 0:
    od[sentence] = n_words


token_dict = od
max_len = max_words

try:
  if max(token_dict.values()) > max_len:
    raise Exception(f"Largest sentence token is {max(token_dict.values())} word tokens long, greater than the permitted maximum of {max_len}.")
except Exception as e:
  print(token_dict)
  print(e)
if len(token_dict) < 2:
  print([token_dict])
total_len = sum(token_dict.values())
min_n_chunks = math.ceil(total_len / max_len)
target_chunk_len = int(round(total_len / min_n_chunks, 0))
new_dict = OrderedDict()
counter = 0
while counter < target_chunk_len:
  if next(iter(token_dict.values())) + counter > max_len:
    break
  sentence, token_len = token_dict.popitem(last=False)
  counter += token_len
  new_dict[sentence] = token_len

In [ ]:
new_dict

In [105]:
token_dict

OrderedDict()

In [106]:
if token_dict:
    result = [new_dict] + split_token_dict(token_dict, max_len)

In [63]:
recent_innovation_df['Split Responses'] = recent_innovation_df['Combined Responses'].apply(lambda response: chunk_text_sentencewise(response, 512-n_prompt_tokens))

ValueError: max() iterable argument is empty

In [31]:
# recent_innovation_df['Prompt Len'] = recent_innovation_df['Prompt'].apply(lambda response: len(tokenizer.tokenize(response)))
# max(recent_innovation_df['Prompt Len'])

In [35]:
responses_as_prompts = [f"{response}. {prompt}" for response in recent_innovation_df['Response'].to_list()]
all_outputs = pipe(responses_as_prompts)

RuntimeError: The size of tensor a (634) must match the size of tensor b (512) at non-singleton dimension 1

In [45]:
token_dict = {'first': 5, 'second': 6}
total_len = sum(token_dict.values())
max_len = 7
int(total_len/max_len) + 1

1

In [59]:
tokenizer.tokenize(" ")

[]